# qlib benchmark run — free GPU (Kaggle / Colab)

Runs `scripts/gpu_run.py` against the private `Riyu-D1/quant-research` repo.

**Setup once:** store a GitHub PAT (read+write on the repo) as secret `GH_TOKEN`
- Kaggle: *Add-ons → Secrets*
- Colab: *key icon → Secrets*

Enable a GPU accelerator before running: Kaggle *Settings → Accelerator (T4/P100)*, Colab *Runtime → Change runtime type → T4*.

US models (`*_us`) need the Alpaca-built qlib dump, published as workflow
artifact `us-qlib-dump` by the `us-alpaca-data` GitHub Action. The fetch cell
below downloads it via the GitHub API using `GH_TOKEN` — **no Alpaca keys in
this notebook**.

In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
WORK = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.path.expanduser('~/cloud-run'))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local', '| workdir:', WORK)

def get_secret(name):
    """GH_TOKEN etc. — Kaggle secrets, Colab userdata, or plain env var."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try:
            return UserSecretsClient().get_secret(name)
        except Exception:
            return os.environ.get(name)
    if ON_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return os.environ.get(name)
    return os.environ.get(name)


In [ ]:
GH_TOKEN = get_secret('GH_TOKEN')
if not GH_TOKEN:
    raise RuntimeError('Set GH_TOKEN secret (PAT with repo access)')
os.environ['GH_TOKEN'] = GH_TOKEN   # phase4 news scorer reads it from env
REPO_URL = f'https://{GH_TOKEN}@github.com/Riyu-D1/quant-research.git'
BRANCH = 'phase6/benchmarks'   # phase-6 GPU wf registry/specs live here
if not os.path.exists('quant-research'):
    !git clone --branch $BRANCH $REPO_URL
# idempotent: re-running cells after a kernel restart must not fail on %cd
if os.path.basename(os.getcwd()) != 'quant-research':
    %cd quant-research


In [ ]:
!pip -q install uv
!uv python install 3.11
!uv sync --frozen --no-dev
import sys, subprocess
print(subprocess.run(['.venv/bin/python', '-c',
    'import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())'],
    capture_output=True, text=True).stdout)

In [ ]:
# Fetch the US qlib dump (artifact from the us-alpaca-data workflow) when
# any *_us model is requested. Uses GH_TOKEN against the GitHub API —
# works on a private repo where `gh run download` would also need auth.
US_DUMP = 'data/qlib/us_alpaca'
# Phase-2 names end in _wf (walk-forward OOF preds into results/oof/<mkt>/);
# phase1 names (transformer_cn hist_cn transformer_us) still work too.
# PHASE 6: one combined GPU session — the four remaining published
# CN walk-forward models (feat20 FilterCol specs in the yaml) plus
# TCN on the US grid. --train-holdout inside each registry entry
# trains OOF folds AND the holdout fold in the same pass.
MODELS = 'tcn_cn_wf alstm_cn_wf tra_cn_wf igmtf_cn_wf tcn_us_wf'  # edit me
# US entries are marked by an '_us' component anywhere in the name —
# transformer_us, hist_us_wf, hist_us_ho, phase2_us — not just a suffix.
if any('_us' in m for m in MODELS.split()) \
        and not os.path.exists(f'{US_DUMP}/calendars/day.txt'):
    import io, json, zipfile, urllib.request
    api = 'https://api.github.com/repos/Riyu-D1/quant-research'
    hdr = {'Authorization': f'Bearer {GH_TOKEN}',
           'Accept': 'application/vnd.github+json'}
    req = urllib.request.Request(
        f'{api}/actions/artifacts?name=us-qlib-dump&per_page=100', headers=hdr)
    arts = [a for a in json.load(urllib.request.urlopen(req))['artifacts']
            if not a['expired']]
    if arts:
        url = sorted(arts, key=lambda a: a['created_at'])[-1]\
            ['archive_download_url']
        # urllib re-sends Authorization on the blob-storage redirect → 401;
        # requests drops it cross-host
        import requests
        r = requests.get(url, headers=hdr, timeout=600)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        os.makedirs(US_DUMP, exist_ok=True)
        z.extractall(US_DUMP)
        print('extracted us-qlib-dump ->', US_DUMP)
    else:
        # a missing dump must not kill the whole session before the push —
        # warn and drop the *_us entries; CN models still train + push.
        drop = [m for m in MODELS.split() if '_us' in m]
        print('WARNING: no us-qlib-dump artifact (run us-alpaca-data first) '
              f'— skipping *_us models: {drop}')
        MODELS = ' '.join(m for m in MODELS.split() if '_us' not in m)
else:
    print('us dump present or not needed')


In [ ]:
!.venv/bin/python scripts/gpu_run.py --list

In [ ]:
# --- results-branch plumbing ------------------------------------------------
# Preds are the point of the run: ckpt() pushes them to RESULTS_BRANCH after
# every model and on a timer, so a crash/disconnect never loses finished
# folds. restore_preds() pulls a prior session's preds back so phase2_run's
# per-fold skip resumes instead of retraining.
import datetime, glob, os, subprocess, threading, time

RESULTS_BRANCH = (get_secret('RESULTS_BRANCH') or
                  f"results/{'kaggle' if ON_KAGGLE else 'colab'}-wf-"
                  f"{datetime.datetime.now():%Y-%m-%d-%H%M}")
# scoped, force-added (-f: results paths may sit under .gitignore on some
# branches); never `git add results/` wholesale — a run that clobbers
# results/LEADERBOARD.md must not get staged.
PUSH_PATHS = ['results/oof', 'results/phase2_cn_holdout_preds.json',
              'results/runs']
CKPT_EVERY = 420  # seconds between background pushes

def _git(*args):
    return subprocess.run(['git', *args], capture_output=True, text=True)

def stage():
    for p in PUSH_PATHS:
        if os.path.exists(p):
            r = _git('add', '-f', p)
            if r.returncode:
                print(f'git add -f {p} FAILED: {r.stderr.strip()}')

def restore_preds():
    """Pull prior pushed preds into the worktree so the per-fold skip sees
    them. No-op if the results branch doesn't exist yet."""
    _git('fetch', 'origin', RESULTS_BRANCH)
    if _git('rev-parse', '--verify', 'FETCH_HEAD').returncode != 0:
        print('no results branch yet — fresh start')
        return 0
    for p in PUSH_PATHS:
        _git('restore', '--source=FETCH_HEAD', '--worktree', '--', p)
    got = len(glob.glob('results/oof/*/*__*.pkl'))
    print(f'restored {got} pred file(s) from origin/{RESULTS_BRANCH}')
    return got

def ckpt(tag=''):
    """Commit staged results on HEAD and push HEAD:<RESULTS_BRANCH>.
    force-with-lease fallback: a second session's base can differ (results
    files are additive — safe). Returns push rc."""
    stage()
    if _git('diff', '--cached', '--quiet').returncode == 0:
        print('ckpt: nothing new', tag)
        return 0
    _git('-c', 'user.name=gpu-run', '-c',
         'user.email=noreply@users.noreply.github.com',
         'commit', '-qm', f'gpu preds {tag}'.strip())
    r = _git('push', '-q', 'origin', f'HEAD:{RESULTS_BRANCH}')
    if r.returncode:
        _git('fetch', 'origin', RESULTS_BRANCH)
        r = _git('push', '-q', '--force-with-lease', 'origin',
                 f'HEAD:{RESULTS_BRANCH}')
    print('ckpt pushed ->', RESULTS_BRANCH, tag, '| rc', r.returncode)
    return r.returncode


In [ ]:
# --- restore any prior preds (idempotent; safe on first run) --------------
restore_preds()
print('preds present:', len(glob.glob('results/oof/*/*__*.pkl')))


In [ ]:
# --- train: one subprocess per model, ckpt after each + periodic push ------
stop = threading.Event()
fails = {}

def _ckpt_loop():
    while not stop.wait(CKPT_EVERY):
        ckpt('(periodic)')

def run_model(name):
    t0 = time.time()
    print(f'=== {name} start ({time.strftime("%H:%M:%S")})', flush=True)
    rc = subprocess.run(['.venv/bin/python', 'scripts/gpu_run.py',
                         '--models', name]).returncode
    mins = (time.time() - t0) / 60
    if rc:
        fails[name] = rc
        print(f'=== {name} FAILED rc={rc} after {mins:.0f}m — continuing',
              flush=True)
    else:
        print(f'=== {name} done in {mins:.0f}m', flush=True)
    ckpt(name)

_th = threading.Thread(target=_ckpt_loop, daemon=True)
_th.start()
try:
    for m in MODELS.split():
        run_model(m)
finally:
    stop.set()
    ckpt('(final)')

done = sorted(glob.glob('results/oof/*/*__*.pkl'))
print(f'\n{len(done)} pred files on disk; failures: {fails or "none"}')


In [ ]:
!cat results/LEADERBOARD.md 2>/dev/null; ls -la results/runs/

In [ ]:
# --- final push: stage everything, verify, push -----------------------------
# After a successful training run something MUST be staged or already
# committed under PUSH_PATHS — the kaggle-wf-2026-11-16 run's final cell
# committed without staging and pushed an empty tip. Fail loudly instead.
stage()
if _git('diff', '--cached', '--quiet').returncode == 0:
    have = _git('ls-files', '--', *PUSH_PATHS).stdout.split()
    if not have:
        raise RuntimeError('training ran but nothing was staged or committed '
                           f'under {PUSH_PATHS} — refusing to push an empty '
                           'branch')
    print('final: nothing new — per-model ckpts already covered '
          f'({len(have)} files on {RESULTS_BRANCH})')
else:
    r = _git('-c', 'user.name=gpu-run', '-c',
             'user.email=noreply@users.noreply.github.com',
             'commit', '-qm', 'gpu run results (final)')
    if r.returncode:
        raise RuntimeError(f'git commit failed: {r.stderr.strip()}')
    r = _git('push', '-q', 'origin', f'HEAD:{RESULTS_BRANCH}')
    if r.returncode:
        _git('fetch', 'origin', RESULTS_BRANCH)
        r = _git('push', '-q', '--force-with-lease', 'origin',
                 f'HEAD:{RESULTS_BRANCH}')
    if r.returncode:
        raise RuntimeError(f'git push failed: {r.stderr.strip()}')
    print('final push ->', RESULTS_BRANCH)

print('\n' + (_git('log', '--oneline', '-5',
                   f'origin/{RESULTS_BRANCH}').stdout or '(not pushed yet)'))
